# Task 3 - SARIMA (Kaggle/Colab)
Run SARIMA for the three required areas and save predictions + metrics.

In [ ]:
# Kaggle-only setup + low-RAM data loading
# This version is optimized for Kaggle and uses your exact dataset path.
from pathlib import Path
import time
import numpy as np
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX
import pyarrow.dataset as ds

DATA_PATH = Path("/kaggle/input/datasets/miraclenanenmbanaade/telecom-traffic-parquet/city_traffic.parquet")
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Parquet not found at {DATA_PATH}")

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Canonical column names expected by forecasting notebooks.
SQUARE_COL = "square_id"
TIME_COL = "time_interval"
TRAFFIC_COL = "internet_traffic"

# Fixed task split for train/test.
TRAIN_END = pd.Timestamp("2013-12-15 23:50:00", tz="UTC")
TEST_START = pd.Timestamp("2013-12-16 00:00:00", tz="UTC")
TEST_END = pd.Timestamp("2013-12-22 23:50:00", tz="UTC")

# Use fixed squares to avoid expensive full-dataset scans.
TARGET_SQUARES = [5161, 4159, 4556]
print(f"Using target squares: {TARGET_SQUARES}")


def build_compact_frame(parquet_path: Path, target_squares: list[int], batch_size: int = 200_000) -> pd.DataFrame:
    # Read only required columns/rows from parquet and aggregate duplicated timestamps.
    dataset = ds.dataset(str(parquet_path), format="parquet")
    square_filter = ds.field(SQUARE_COL).isin([int(s) for s in target_squares])

    scanner = dataset.scanner(
        columns=[SQUARE_COL, TIME_COL, TRAFFIC_COL],
        filter=square_filter,
        batch_size=batch_size,
    )

    agg = {}
    for batch in scanner.to_batches():
        pdf = batch.to_pandas()
        if pdf.empty:
            continue

        # Normalize and clean values before aggregation.
        pdf[SQUARE_COL] = pd.to_numeric(pdf[SQUARE_COL], errors="coerce")
        pdf[TIME_COL] = pd.to_datetime(pdf[TIME_COL], utc=True, errors="coerce")
        pdf[TRAFFIC_COL] = pd.to_numeric(pdf[TRAFFIC_COL], errors="coerce")
        pdf = pdf.replace([np.inf, -np.inf], np.nan).dropna(subset=[SQUARE_COL, TIME_COL, TRAFFIC_COL])

        grouped = pdf.groupby([SQUARE_COL, TIME_COL], sort=False)[TRAFFIC_COL].sum()
        for (sid, ts), val in grouped.items():
            key = (int(sid), pd.Timestamp(ts))
            agg[key] = agg.get(key, 0.0) + float(val)

    if not agg:
        raise ValueError("No rows found for selected squares")

    rows = [{SQUARE_COL: sid, TIME_COL: ts, TRAFFIC_COL: val} for (sid, ts), val in agg.items()]
    compact_df = pd.DataFrame(rows).sort_values([SQUARE_COL, TIME_COL]).reset_index(drop=True)
    return compact_df


print("Loading compact dataframe from parquet...")
df = build_compact_frame(DATA_PATH, TARGET_SQUARES)
print(f"Compact rows loaded: {len(df):,}")


def get_area_series(data: pd.DataFrame, square_id: int) -> pd.Series:
    s = data[data[SQUARE_COL] == int(square_id)].sort_values(TIME_COL)
    series = pd.to_numeric(s.set_index(TIME_COL)[TRAFFIC_COL], errors="coerce").astype(float)
    series = series.replace([np.inf, -np.inf], np.nan)

    # Enforce fixed 10-minute grid for stable SARIMA date handling.
    series = series.resample("10min").mean()
    series = series.interpolate(limit_direction="both")
    series = series.ffill().bfill().dropna()
    return series


def split_train_test(series: pd.Series):
    train = series.loc[:TRAIN_END]
    test = series.loc[TEST_START:TEST_END]
    return train.dropna(), test.dropna()

target_squares = TARGET_SQUARES

In [ ]:
# SARIMA training + evaluation
# Settings are chosen to balance speed and acceptable forecast quality on Kaggle.
def evaluate_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    y_true = np.asarray(y_true, dtype=float).reshape(-1)
    y_pred = np.asarray(y_pred, dtype=float).reshape(-1)
    n = min(len(y_true), len(y_pred))
    y_true = y_true[:n]
    y_pred = y_pred[:n]
    mae = float(np.mean(np.abs(y_true - y_pred)))
    mape = float(np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), 1e-8))) * 100.0)
    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    return {"MAE": mae, "MAPE": mape, "RMSE": rmse}

ORDER = (1, 1, 1)
SEASONAL_ORDER = (1, 0, 1, 144)
MAXITER = 30

sarima_results = []
for square_id in target_squares:
    series = get_area_series(df, int(square_id))
    train, test = split_train_test(series)

    if len(train) < 3 * 144 or len(test) == 0:
        print(f"Skipping square {square_id}: insufficient train/test points")
        continue

    start = time.perf_counter()
    model = SARIMAX(
        train,
        order=ORDER,
        seasonal_order=SEASONAL_ORDER,
        enforce_stationarity=False,
        enforce_invertibility=False,
    )
    fit = model.fit(disp=False, maxiter=MAXITER)
    train_time = time.perf_counter() - start

    start = time.perf_counter()
    forecast = fit.forecast(steps=len(test))
    infer_time = time.perf_counter() - start

    y_true = test.values
    y_pred = np.asarray(forecast.values)
    metrics = evaluate_metrics(y_true, y_pred)
    metrics.update({"square_id": int(square_id), "train_seconds": train_time, "infer_seconds": infer_time})
    sarima_results.append(metrics)
    print(f"Square {square_id} done in {train_time:.1f}s (train), {infer_time:.2f}s (forecast)")

    out = pd.DataFrame({
        "time_interval": test.index[: len(y_pred)],
        "y_true": y_true[: len(y_pred)],
        "y_pred": y_pred,
        "model": "SARIMA",
        "square_id": int(square_id),
    })
    out.to_csv(OUTPUT_DIR / f"sarima_{int(square_id)}.csv", index=False)

metrics_df = pd.DataFrame(sarima_results)
metrics_df.to_csv(OUTPUT_DIR / "metrics_sarima.csv", index=False)
print(f"Saved outputs to: {OUTPUT_DIR.resolve()}")
metrics_df